# 07 · A paged KV-cache allocator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/inference/07_paged_kv_allocator.ipynb) &nbsp; 📖 Concept deck: [PagedAttention and prefix caching](https://nilesh-salpe.github.io/visual-ai/inference/05-paged-attention/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** the memory manager behind PagedAttention (vLLM, SOSP 2023): fixed-size blocks, per-sequence block tables, reference counts, and copy-on-write when forked sequences diverge.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")

print('no extra setup')

## Task 1: BlockManager

Implement the class with `num_blocks` physical blocks of `block_size` token slots each:
- `add(seq)`: register an empty sequence.
- `append(seq, n=1)`: add `n` tokens one at a time. Allocate a new block when the sequence is empty or its last block is full. If the last block is **shared** (refcount > 1) and not full, **copy-on-write**: allocate a fresh block, point the table at it, and decrement the old block's refcount. Raise `MemoryError` when no block is free.
- `fork(parent, child)`: the child shares all of the parent's blocks (refcounts +1) and gets the same token count.
- `free(seq)`: decrement refcounts and return blocks that reach 0 to the free pool.
- `num_free()`, `table(seq)` (list of physical block ids), `num_tokens(seq)`.

In [ ]:
class BlockManager:
    def __init__(self, num_blocks, block_size=16):
        # TODO
        raise NotImplementedError
    def add(self, seq): raise NotImplementedError
    def append(self, seq, n=1): raise NotImplementedError
    def fork(self, parent, child): raise NotImplementedError
    def free(self, seq): raise NotImplementedError
    def num_free(self): raise NotImplementedError
    def table(self, seq): raise NotImplementedError
    def num_tokens(self, seq): raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_BlockManager(C):
    m = C(10, block_size=4)
    m.add('a'); m.append('a', 9)
    assert len(m.table('a')) == 3 and m.num_free() == 7, "9 tokens with block_size 4 need 3 blocks"
    m.fork('a', 'b')
    assert m.table('b') == m.table('a') and m.num_free() == 7, "fork must share blocks, not copy them"
    m.append('b')
    assert m.table('b')[:2] == m.table('a')[:2], "full blocks stay shared after a write"
    assert m.table('b')[2] != m.table('a')[2] and m.num_free() == 6, "writing into a shared partial block must copy-on-write"
    assert m.num_tokens('b') == 10 and m.num_tokens('a') == 9
    m.free('a')
    assert m.num_free() == 7, "freeing 'a' releases only its unshared last block"
    m.free('b')
    assert m.num_free() == 10, "everything returned after freeing both"
    m2 = C(2, block_size=2); m2.add('x'); m2.append('x', 4)
    try:
        m2.append('x'); raise AssertionError("expected MemoryError when out of blocks")
    except MemoryError:
        pass

check("BlockManager", _test_BlockManager, BlockManager)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: BlockManager
class BlockManager_solution:
    def __init__(self, num_blocks, block_size=16):
        self.bs, self.free_ids = block_size, list(range(num_blocks))[::-1]
        self.ref, self.tables, self.ntok = [0] * num_blocks, {}, {}
    def _alloc(self):
        if not self.free_ids: raise MemoryError("out of KV blocks")
        b = self.free_ids.pop(); self.ref[b] = 1; return b
    def _release(self, b):
        self.ref[b] -= 1
        if self.ref[b] == 0: self.free_ids.append(b)
    def add(self, seq): self.tables[seq], self.ntok[seq] = [], 0
    def append(self, seq, n=1):
        for _ in range(n):
            t, k = self.tables[seq], self.ntok[seq]
            if k % self.bs == 0:                                  # empty or last block full
                t.append(self._alloc())
            elif self.ref[t[-1]] > 1:                             # shared partial block -> copy-on-write
                new = self._alloc(); self._release(t[-1]); t[-1] = new
            self.ntok[seq] = k + 1
    def fork(self, parent, child):
        self.tables[child] = list(self.tables[parent]); self.ntok[child] = self.ntok[parent]
        for b in self.tables[child]: self.ref[b] += 1
    def free(self, seq):
        for b in self.tables.pop(seq): self._release(b)
        del self.ntok[seq]
    def num_free(self): return len(self.free_ids)
    def table(self, seq): return list(self.tables[seq])
    def num_tokens(self, seq): return self.ntok[seq]

check("BlockManager (reference)", _test_BlockManager, BlockManager_solution)

---
## Experiments
These use the reference solutions, so they run even before you finish.

In [ ]:
# Parallel sampling: 4 completions share one 600-token prompt.
m = BlockManager_solution(400, 16); m.add('prompt'); m.append('prompt', 600)
for i in range(4):
    m.fork('prompt', f's{i}'); m.append(f's{i}', 100)
used = 400 - m.num_free()
print(f"shared prompt + 4 x 100-token samples use {used} blocks; without sharing it would be {4 * math.ceil(700 / 16)} blocks")

---
## Takeaways
- Blocks plus a per-sequence **block table** let the KV cache grow without contiguous reservations, wasting less than one block per sequence.
- **Reference counts and copy-on-write** let parallel samples and beam candidates share their prompt's KV memory.
- vLLM's default block size is **16** tokens. It reported **2–4×** throughput over FasterTransformer and Orca at the same latency.